# Analisis Sentimen IndoBERTweet - **Baseline**
> **Model:** `indolem/indobertweet-base-uncased` | **Dataset:** Tweet RKUHAP | **Loss:** Cross-Entropy (default)

Eksperimen baseline - tidak ada penanganan class imbalance. Digunakan sebagai acuan perbandingan terhadap tiga strategi lainnya.

## 1. Setup
Install library dan import semua dependensi yang dibutuhkan.

In [ ]:
!pip install transformers datasets accelerate evaluate wordcloud seaborn PySastrawi -q

import re, random, os, zipfile, pickle
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud, STOPWORDS
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback
)
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

## 2. Load & Preprocessing
Dataset: Tweet RKUHAP dari GitHub.  
- `text_model` - teks bersih untuk input model (hapus URL, normalisasi spasi).  
- `wc_text` - teks untuk WordCloud (huruf kecil, hanya alfabet).

In [ ]:
DATA_URL = (
    "https://raw.githubusercontent.com/error404-sudo/"
    "Analisis-Sentimen-indoBERTTweet/main/"
    "Analisis-Sentimen-RKUHAP-FinalLabel.csv"
)
df = pd.read_csv(DATA_URL)
print(f"Jumlah data : {len(df)} | Kolom: {df.columns.tolist()}")
print(f"Missing     : {df.isnull().sum().sum()} | Duplikasi: {df.duplicated().sum()}")

# text_model: hapus URL + normalisasi spasi
df["text_model"] = df["full_text"].apply(
    lambda x: re.sub(r"\s+", " ",
        re.sub(r"https?://\S+|www\.\S+", "", str(x))
        .replace("\n", " ").replace("\r", " ")
    ).strip()
)
# wc_text: untuk EDA & WordCloud
df["wc_text"] = df["full_text"].apply(
    lambda x: " ".join(
        w for w in re.sub(r"[^a-zA-Z\s]", " ", str(x).lower()).split()
        if len(w) > 2
    )
)
print("Preprocessing selesai.")

## 3. EDA & WordCloud
Visualisasi distribusi kelas, panjang teks, dan kata dominan per sentimen.

In [ ]:
COLORS = {"positif": "#2a78d6", "neutral": "#1baf7a", "negatif": "#e34948"}
ORDER  = ["positif", "neutral", "negatif"]
vc     = df["sentimen"].value_counts()
print(f"Imbalance Ratio: {vc.max()/vc.min():.2f}x | {vc.to_dict()}")

# Gambar 1: Proporsi Kelas Sentimen
fig, ax = plt.subplots(figsize=(6, 4))
counts = [vc.get(k, 0) for k in ORDER]
total  = sum(counts)
bars   = ax.bar(
    [k.capitalize() for k in ORDER], counts,
    color=[COLORS[k] for k in ORDER], width=0.5, linewidth=0, zorder=3
)
for bar, n in zip(bars, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + total * 0.01,
        f"{n} · {n/total*100:.1f}%", ha="center", va="bottom", fontsize=10
    )
ax.set_title("Proporsi Kelas Sentimen", fontweight="bold")
ax.set_ylabel("Jumlah Teks")
ax.set_ylim(0, max(counts) * 1.15)
ax.grid(axis="y", linewidth=0.5, alpha=0.4, zorder=0)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

# Gambar 2: Distribusi Panjang Teks
fig, ax = plt.subplots(figsize=(10, 4))
bins_   = [0, 10, 20, 30, 40, 50, 75, 100, 150, np.inf]
xlabels = ["1–10","11–20","21–30","31–40","41–50","51–75","76–100","101–150","151+"]
x, w   = np.arange(len(xlabels)), 0.25
for i, sent in enumerate(ORDER):
    lengths = df[df["sentimen"] == sent]["text_model"].str.split().str.len()
    cnt, _  = np.histogram(lengths, bins=bins_)
    ax.bar(x + i*w, cnt, width=w, label=sent.capitalize(),
           color=COLORS[sent], linewidth=0, zorder=3)
ax.set_xticks(x + w); ax.set_xticklabels(xlabels, fontsize=9)
ax.set_title("Distribusi Panjang Teks", fontweight="bold")
ax.set_xlabel("Jumlah Kata"); ax.set_ylabel("Frekuensi")
ax.legend(frameon=False)
ax.grid(axis="y", linewidth=0.5, alpha=0.4, zorder=0)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

# Stopword
_sw = set(StopWordRemoverFactory().get_stop_words())
SW  = _sw | STOPWORDS | {
    "aja","nih","sih","deh","dong","kok","kan","lah","mah","pun","nya","ya","tuh",
    "kalo","gini","banget","bgt","cuma","mau","buat","lagi","udah","bakal","biar",
    "sama","orang","mereka","kita","saya","gue","gua","gw","lu","lo","elu","bro",
    "min","wkwk","wkwkwk","haha","hehe","https","http","co","com","amp","rt","tco",
    "www","rkuhap","kuhap","ruu","uu","undang","undangundang","hukum","pasal",
    "pengesahan","mengesahkan","disahkan","sahkan","sah","rancangan","kitab","acara",
    "pidana","kuhp","indonesia","negara","pemerintah","dpr","ri","komisi","iii",
    "paripurna","presiden","ketua","anggota","fraksi","resmi","rapat","sidang",
    "pembahasan","proses","hari","tahun","november","selasa","kemarin","besok",
    "baru","lama","aturan","isu","soal","berita","narasi","pamflet","klaim",
    "fakta","faktanya","bilang","baca","bahas","poin","isinya","beredar"
}
df["wc_text"] = df["wc_text"].apply(lambda x: " ".join(w for w in x.split() if w not in SW))

# WordCloud per Sentimen
WC_CMAPS = {"positif": "Blues", "neutral": "Greens", "negatif": "Reds"}
for sent in ORDER:
    fig, ax = plt.subplots(figsize=(8, 5))
    text = " ".join(df[df["sentimen"] == sent]["wc_text"])
    wc   = WordCloud(
        width=800, height=500, background_color="white",
        max_words=80, colormap=WC_CMAPS[sent],
        stopwords=SW, collocations=False, random_state=42
    ).generate(text)
    ax.imshow(wc, interpolation="bilinear"); ax.axis("off")
    ax.set_title(f"WordCloud — {sent.capitalize()}", fontweight="bold", fontsize=13)
    plt.tight_layout(); plt.show()

## 4. Label Encoding & Split Data
Stratified split **70 / 15 / 15** - `random_state=42`.

In [ ]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["sentimen"])
print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

X_train, X_temp, y_train, y_temp = train_test_split(
    df["text_model"], df["label"], test_size=0.30, random_state=42, stratify=df["label"]
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)
id2label     = dict(enumerate(le.classes_))
label_counts = lambda y: pd.Series(y).map(id2label).value_counts().reindex(le.classes_, fill_value=0)
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(
    pd.DataFrame({
        "Train": label_counts(y_train),
        "Val"  : label_counts(y_val),
        "Test" : label_counts(y_test)
    }).T.to_string()
)

## 5. Tokenisasi & Dataset
`max_length=128` | Training menggunakan data asli (tanpa oversampling).

In [ ]:
MODEL_NAME = "indolem/indobertweet-base-uncased"
MAX_LENGTH = 128
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item           = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

def tokenize(texts):
    return tokenizer(texts.tolist(), truncation=True, padding=True, max_length=MAX_LENGTH)


train_enc     = tokenize(X_train)
val_enc       = tokenize(X_val)
test_enc      = tokenize(X_test)
train_dataset = SentimentDataset(train_enc, y_train)
val_dataset   = SentimentDataset(val_enc,   y_val)
test_dataset  = SentimentDataset(test_enc,  y_test)
print(f"Dataset siap — Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")


## 6. Training
**Loss:** Cross-Entropy default | Tidak ada class weight maupun oversampling.

In [ ]:
def compute_metrics(eval_pred):
    preds       = eval_pred[0].argmax(axis=1)
    labels      = eval_pred[1]
    report      = classification_report(labels, preds, output_dict=True, zero_division=0)
    neutral_idx = str(le.transform(["neutral"])[0])
    return {
        "accuracy"   : accuracy_score(labels, preds),
        "macro_f1"   : report["macro avg"]["f1-score"],
        "weighted_f1": report["weighted avg"]["f1-score"],
        "neutral_f1" : report[neutral_idx]["f1-score"],
    }

OUTPUT_DIR = "./hasil_baseline"
model   = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(le.classes_))
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        eval_strategy="epoch", save_strategy="epoch", logging_strategy="epoch",
        num_train_epochs=5, learning_rate=2e-5,
        per_device_train_batch_size=32, per_device_eval_batch_size=16,
        weight_decay=0.01, load_best_model_at_end=True,
        metric_for_best_model="macro_f1", greater_is_better=True,
        save_total_limit=2, report_to="none", seed=42
    ),
    train_dataset=train_dataset, eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)
trainer.train()


## 7. Grafik & Ringkasan Epoch

In [ ]:
logs       = trainer.state.log_history
train_loss = [(round(l["epoch"]), l["loss"])          for l in logs if "loss"          in l and "eval_loss" not in l]
val_loss   = [(round(l["epoch"]), l["eval_loss"])     for l in logs if "eval_loss"     in l]
val_f1     = [(round(l["epoch"]), l["eval_macro_f1"]) for l in logs if "eval_macro_f1" in l]

epochs   = [e for e, _ in val_loss]
fig, ax1 = plt.subplots(figsize=(12, 5))
ax2      = ax1.twinx()
ax1.plot(*zip(*train_loss), "o-", label="Train Loss",      color="#E74C3C")
ax1.plot(*zip(*val_loss),   "s-", label="Validation Loss", color="#3498DB")
ax2.plot(*zip(*val_f1),     "D-", label="Val Macro F1",    color="#2ECC71")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax2.set_ylabel("Macro F1")
ax1.set_xticks(epochs)
ax1.set_title("Training Loss & Validation Macro F1 per Epoch", fontweight="bold")
lines1, lbl1 = ax1.get_legend_handles_labels()
lines2, lbl2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, lbl1 + lbl2, loc="upper right")
ax1.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Ringkasan per epoch
epoch_train = {round(l["epoch"]): l["loss"] for l in logs if "loss" in l and "eval_loss" not in l}
print(f"{'Epoch':>6}  {'Train Loss':>10}  {'Val Loss':>9}  {'Val Acc':>8}  {'Macro F1':>9}")
print("-" * 52)
for l in logs:
    if "eval_loss" in l:
        ep = round(l["epoch"])
        print(
            f"{ep:>6}  {epoch_train.get(ep, float('nan')):>10.4f}"
            f"  {l.get('eval_loss',     float('nan')):>9.4f}"
            f"  {l.get('eval_accuracy', float('nan')):>8.4f}"
            f"  {l.get('eval_macro_f1', float('nan')):>9.4f}"
        )

## 8. Evaluasi Test Set

In [ ]:
predictions = trainer.predict(test_dataset)
y_pred      = predictions.predictions.argmax(axis=1)

print("=" * 55)
print(f"  Accuracy          : {accuracy_score(y_test, y_pred):.4f}")
print(f"  Precision (macro) : {precision_score(y_test, y_pred, average='macro',    zero_division=0):.4f}")
print(f"  Recall    (macro) : {recall_score(   y_test, y_pred, average='macro',    zero_division=0):.4f}")
print(f"  Macro F1          : {f1_score(       y_test, y_pred, average='macro',    zero_division=0):.4f}")
print(f"  Weighted F1       : {f1_score(       y_test, y_pred, average='weighted', zero_division=0):.4f}")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion Matrix — Baseline", fontweight="bold")
plt.tight_layout(); plt.show()


## 9. Prediksi Teks Baru

In [ ]:
def predict_sentiment(text):
    model.eval()
    enc = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=MAX_LENGTH, padding=True
    )
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits, dim=1).squeeze().cpu().numpy()
    pred = le.classes_[probs.argmax()]
    print(f"Teks    : {text}")
    print(f"Prediksi: {pred.upper()}")
    for lbl, p in zip(le.classes_, probs):
        print(f"  {lbl:>8}: {p:.4f}  {'█'*int(p*30)}")
    return pred

SAMPLE_TEXTS = [
    "RKUHAP ini sangat baik dan progresif untuk reformasi hukum kita!",
    "Aturan ini tidak jelas dan merugikan banyak pihak.",
    "Sidang paripurna DPR membahas RKUHAP hari ini.",
]
for t in SAMPLE_TEXTS:
    print("=" * 60); predict_sentiment(t)

## 10. Ringkasan Konfigurasi

In [ ]:
print("=" * 55)
print("  RINGKASAN KONFIGURASI EKSPERIMEN")
print("=" * 55)
cfg = {
    "Metode"               : "Baseline",
    "Model"                : MODEL_NAME,
    "Learning Rate"        : "2e-5",
    "Batch Size (Train)"   : 32,
    "Batch Size (Eval)"    : 16,
    "Max Epoch"            : 5,
    "Max Length"           : MAX_LENGTH,
    "Weight Decay"         : 0.01,
    "Seed"                 : 42,
    "Penanganan Imbalance" : "Tidak ada (data asli)",
}
for k, v in cfg.items():
    print(f"  {k:<24}: {v}")
print("=" * 55)


## 11. Simpan & Unduh Model
Model, tokenizer, dan label encoder dikemas dalam satu file `.zip`.

In [ ]:
SAVE_DIR = "./saved_model"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
with open(f"{SAVE_DIR}/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

ZIP_PATH = "./saved_model.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(SAVE_DIR):
        for file in files:
            fp = os.path.join(root, file)
            zf.write(fp, os.path.relpath(fp, SAVE_DIR))

print(f"Model tersimpan : {SAVE_DIR}")
print(f"Zip siap unduh  : {ZIP_PATH}")
print(f"Ukuran zip      : {os.path.getsize(ZIP_PATH)/1e6:.1f} MB")

from google.colab import files
files.download(ZIP_PATH)